In [1]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns

df = pd.read_csv("vulkan_knjige_obradjeno.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21410 entries, 0 to 21409
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   naslov           21410 non-null  object 
 1   autor            21410 non-null  object 
 2   zanr             21410 non-null  object 
 3   izdavac          21410 non-null  object 
 4   godina           21410 non-null  int64  
 5   br_strana        21410 non-null  int64  
 6   cena             21410 non-null  int64  
 7   povrsina         21410 non-null  float64
 8   papir            21410 non-null  float64
 9   povez_Tvrd       21410 non-null  bool   
 10  pismo_Ćirilica   21410 non-null  bool   
 11  kategorije_cena  21410 non-null  object 
dtypes: bool(2), float64(2), int64(3), object(5)
memory usage: 1.7+ MB


In [2]:
df['materijal'] = df['povrsina'] * df['br_strana']

In [3]:
df['kategorije_cena'].value_counts()

kategorije_cena
500-1000     10446
1000-1500     5378
1500-2000     2036
0-500         1997
2000-2500      690
2500-3000      447
3000-3500      177
3500-4000      131
4000+          108
Name: count, dtype: int64

In [4]:
mapa_cena = {
    "0-500": 0,
    "500-1000": 1,
    "1000-1500": 2,
    "1500-2000": 3,
    "2000-2500": 4,
    "2500-3000": 5,
    "3000-3500": 6,
    "3500-4000": 7,
    "4000+": 8
}

In [5]:
# prevodi raspon cena u ceo broj
def to_numerical(df, kolona):
    print("debug", df[kolona].unique()) # debug step
    df = df.copy()  # ne modifikuje se originalni df zbog greske
    df[kolona] = df[kolona].str.strip().map(mapa_cena) 
    df.loc[df[kolona].isna(), kolona] = -1  # Ako ima Nan
    return df

# ceo broj pretvara u raspon cena
def to_categorical(df, kolona):
    obrni = {v: k for k, v in mapa_cena.items()}
    df = df.copy()  # Avoid chained assignment issues
    df[kolona] = df[kolona].map(obrni)
    return df

In [6]:
df = to_numerical(df, 'kategorije_cena')
df.info()

debug ['0-500' '500-1000' '1000-1500' '1500-2000' '2000-2500' '2500-3000'
 '3000-3500' '3500-4000' '4000+']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21410 entries, 0 to 21409
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   naslov           21410 non-null  object 
 1   autor            21410 non-null  object 
 2   zanr             21410 non-null  object 
 3   izdavac          21410 non-null  object 
 4   godina           21410 non-null  int64  
 5   br_strana        21410 non-null  int64  
 6   cena             21410 non-null  int64  
 7   povrsina         21410 non-null  float64
 8   papir            21410 non-null  float64
 9   povez_Tvrd       21410 non-null  bool   
 10  pismo_Ćirilica   21410 non-null  bool   
 11  kategorije_cena  21410 non-null  int64  
 12  materijal        21410 non-null  float64
dtypes: bool(2), float64(3), int64(4), object(4)
memory usage: 1.8+ MB


Kao nastavak projekta vezan za skup knjiga iz Vulkana, u ovom notebook-u će biti implementirane Multinomijalne Logističke Regresije korišćenjem Softmax funkcije, zatim Bajesov Naivni Klasifikator i klasterovanje pomoću K srednjih vrednosti.
# Klasifikacija
## Logistička regresija
U ovom delu se ovaj dokument bavi razvijanjem Multinomijalne logističke regresije. Naime kao ulaz biće korišćene kao i kod linearne regresije vrednosti promenljivih `materijal` i `povez_Tvrd` sa tim što će na osnovu ovih vrednosti cena knjige biti klasivikovana u okviru jedne od klasa vrednosti `kategorije_cena`.

Sama logistička regresija podrazumeva Bernulijevu raspodelu i u ovom slučaju više klasa. Za razliku od binarne logistiške regresije koja klasifikuje vrednosti na osnovu verovatnoće da vrednost pripada jednoj od dve klase pomoću sigmoidne funkcije, kod višeklasne logističke klasifikacije postoji izbor klasifikacije 1 na 1, 1 na Ostale i multinomijalnom regresijomkoja umesto sigmoidne funkcije se mora koristiti softmax funkcija. Sa obzirom da bi prva i druga koristile više RAM memorije, a ja imam ograničene resurse, fokusiraću se na multinomijalnu regresiju. 

Radi lakše obrade vrednosti će biti podeljene u nekoliko međumatrica:
 - X: to je ulazna matrica pojedinačnih knjiga i atributa `materijal` i `povez_Tvrd`
 - W: matrica težinskih koeficijenata matematički sa atributima opisuje eksponente dok sa druge strane predstavlja vrednost između atributa i klasa
 - Z: predstavlja sam eksponent klase dok matematicki ima oblik matricnog mnozenja: Z = X * W
 - P: predstavlja verovatnoće da pojedine knjige pripadaju pojedinim klasama. Dobijena je korišćenjem softmax funkcije: P = softmax(Z)
 - Y: ova matrica predstavlja izlazne vrednosti odnosno klasu kojoj cena knjiga pripada. Dobija se odabirom najveće verovatnoće iz svakog reda matrice verovatnoća Y = argmax(P)

Pored ove transformacije parametri W se podešavaju (odnosno treniraju) pomoću Cross Entropy Loss funkcijom L = - sum(y * log(z)) sa idejom da sto je entropija veća i nesigurnost verovatnoće je veća pa samim tim i model neprecizniji. Pronalaženjem mimimuma ove funkcije (što se radi numeriški pomoću softvera) dobijaju se vrednosti o tome u kom smeru ide učenje.

U narednom odeljku se radi na razvijanju opisane funkcije.

In [8]:
class MySoftmaxRegression:
    def __init__(self, epoch = 1000, lr = 0.001, min_change = 1e-10, lmbda = 0.001): # constructor contains important global variable
        self.y_true = None # one hot value of y
        self.x_true = None
        self.p = None
        self.y_pred = None
        self.num_class = None
        self.epoch = epoch
        self.lr = lr
        self.prev_cost = 1e10
        self.min_change = min_change
        self.lmbda = lmbda
    
    def preprocess(self, x, y): # inicializes input data
        self.x_true = x
        self.num_class = len(np.unique(y))
        self.y_true = self.one_hot(y)
        self.w = np.zeros((self.x_true.shape[1], self.num_class))
        self.b = np.zeros((1, self.num_class))
        
    def transform(self): # transforms input data
        # transform of x
        z = np.dot(self.x_true, self.w) + self.b
        self.p = self.softmax(z) # ------- axis?
        self.y_pred = np.argmax(self.p, axis=1)
        
    def cross_entropy(self): # calculates gradient

        # Compute gradients
        dw = -np.dot(self.x_true.T, (self.y_true - self.p)) / self.x_true.shape[0] #+ self.lmbda * self.w # for regularization
        db = np.sum((self.p - self.y_true), axis=0) / self.x_true.shape[0]
    
        # Update weights and bias
        self.w -= self.lr * dw
        self.b -= self.lr * db
            
    def cost(self): # calculates loss
        return -(np.sum(self.y_true * np.log(np.clip(self.p, 1e-10, 1)))) / self.x_true.shape[0]
        
    def fit(self, x_train, y_train): #trains model
        self.preprocess(x_train, y_train)
        for epoch in range(self.epoch):
            self.transform()
            if abs(self.prev_cost - self.cost()) < self.min_change:
                print(f"End epoch {epoch} cost is {self.prev_cost}")
                break
            else:
                self.prev_cost = self.cost()
            
            self.cross_entropy()
            
    def predict(self, x_train): # predicts output value based on input
        self.x_true = x_train
        self.transform()
        return self.y_pred
        
    def one_hot(self, y): # gives one hot matrix
        y = np.array(y)  
        y = y.astype(int)
        one_hot_matrix = np.zeros((y.shape[0], self.num_class))
        one_hot_matrix[np.arange(y.shape[0]), y] = 1
    
        return one_hot_matrix
        
    def softmax(self, z): # calculates probability based on softmax formula
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))  # stabilizing by subtracting max
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

In [9]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def compare_models(y_true, y_pred_custom, y_pred_sklearn):
    metrics = {
        "Accuracy": accuracy_score,
        "Precision": lambda y_true, y_pred: precision_score(y_true, y_pred, average='macro', zero_division=0),
        "Recall": lambda y_true, y_pred: recall_score(y_true, y_pred, average='macro'),
        "F1-Score": lambda y_true, y_pred: f1_score(y_true, y_pred, average='macro')
    }
    
    print("\nModel Performance Comparison")
    print("-" * 40)
    print(f"{'Metric':<15} {'My Model':<15} {'Sklearn Model':<15}")
    print("-" * 40)
    
    for name, metric in metrics.items():
        custom_value = metric(y_true, y_pred_custom)
        sklearn_value = metric(y_true, y_pred_sklearn)
        print(f"{name:<15} {custom_value:<15.4f} {sklearn_value:<15.4f}")

    # Confusion Matrices
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    cm_custom = confusion_matrix(y_true, y_pred_custom)
    cm_sklearn = confusion_matrix(y_true, y_pred_sklearn)

    sns.heatmap(cm_custom, annot=True, fmt="d", cmap="Blues", ax=axes[0])
    axes[0].set_title("Confusion Matrix - My Model")
    axes[0].set_xlabel("Predicted Label")
    axes[0].set_ylabel("True Label")

    sns.heatmap(cm_sklearn, annot=True, fmt="d", cmap="Greens", ax=axes[1])
    axes[1].set_title("Confusion Matrix - Sklearn Model")
    axes[1].set_xlabel("Predicted Label")
    axes[1].set_ylabel("True Label")

    plt.tight_layout()
    plt.show()

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X = df[['br_strana', 'povrsina', 'povez_Tvrd', 'godina']].values
y = df['kategorije_cena'].values

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

my_model = MySoftmaxRegression(epoch=5000, lr=0.01)
my_model.fit(X_train, y_train)
y_pred_my = my_model.predict(X_test)

sklearn_model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=5000)
sklearn_model.fit(X_train, y_train)
y_pred_sklearn = sklearn_model.predict(X_test)

In [11]:
compare_models(y_test, y_pred_my, y_pred_sklearn)


Model Performance Comparison
----------------------------------------
Metric          My Model        Sklearn Model  
----------------------------------------
Accuracy        0.5731          0.5838         
Precision       0.2001          0.2555         
Recall          0.1642          0.1923         
F1-Score        0.1583          0.1982         


AttributeError: module 'matplotlib' has no attribute 'subplots'

In [18]:
num_cat = 8

# quantile-based bins
df['kategorije_cena_2'], bins = pd.qcut(df['cena'], q=num_cat, labels=False, retbins=True)
print(df['kategorije_cena_2'].value_counts())
print(bins)

kategorije_cena_2
2    3371
1    2770
3    2764
0    2679
6    2647
7    2646
5    2423
4    2110
Name: count, dtype: int64
[ 249.  566.  715.  880.  990. 1099. 1299. 1699. 5489.]


In [20]:
y = df['kategorije_cena_2'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

my_model = MySoftmaxRegression(epoch=5000, lr=0.01)
my_model.fit(X_train, y_train)
y_pred_my = my_model.predict(X_test)

sklearn_model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=5000)
sklearn_model.fit(X_train, y_train)
y_pred_sklearn = sklearn_model.predict(X_test)

In [21]:
compare_models(y_test, y_pred_my, y_pred_sklearn)


Model Performance Comparison
----------------------------------------
Metric          My Model        Sklearn Model  
----------------------------------------
Accuracy        0.3340          0.3389         
Precision       0.3243          0.3291         
Recall          0.3264          0.3323         
F1-Score        0.2977          0.3127         


AttributeError: module 'matplotlib' has no attribute 'subplots'

### Analiza Performansi Modela  

1. Performanse na balansiranom skupu podataka  
 
- Niska tačnost (~33%) kod oba modela ukazuje na poteškoće u klasifikaciji.  
- Precision, Recall i F1-score su niski, ali uravnoteženi, što znači da modeli ne favorizuju određene klase previše.  

2. Performanse na nebalansiranom skupu podataka
 
- Tačnost raste (~58%), što može biti varljivo zbog pojedinih klasa koje su zastupljenije.  
- Precision i Recall značajno opadaju, što znači da modeli teško prepoznaju manje zastupljene klase.  
- Sklearn model blago nadmašuje moj model u svim metrikama, ali oba imaju nizak recall, što pokazuje da propuštaju mnoge pozitivne slučajeve.  

3. Ključni zaključci

- Balansirani podaci → Pravednija predviđanja: Greške su ravnomernije raspoređene po klasama.  
- Nebalansirani podaci → Pristrasnost ka većinskim klasama: Modeli dobro rade sa dominantnim klasama, ali ignorišu manje zastupljene.  
- Sklearn model je malo bolji.  
- Moguća poboljšanja: Korišćenje tehnika rebalansiranja podataka (oversampling, undersampling).  


# Klasterizacija
## K srednjih vrednosti

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21410 entries, 0 to 21409
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   naslov             21410 non-null  object 
 1   autor              21410 non-null  object 
 2   zanr               21410 non-null  object 
 3   izdavac            21410 non-null  object 
 4   godina             21410 non-null  int64  
 5   br_strana          21410 non-null  int64  
 6   cena               21410 non-null  int64  
 7   povrsina           21410 non-null  float64
 8   papir              21410 non-null  float64
 9   povez_Tvrd         21410 non-null  bool   
 10  pismo_Ćirilica     21410 non-null  bool   
 11  kategorije_cena    21410 non-null  int64  
 12  materijal          21410 non-null  float64
 13  kategorije_cena_2  21410 non-null  int64  
dtypes: bool(2), float64(3), int64(5), object(4)
memory usage: 2.0+ MB


Postupak:
 1. Skaliranje podataka (kako ni jedna od kolona ne bi dominirala)
 2. Dodeljivanje random slučajnih centralnih vrednosti `cent`
 3. Dodeljivanje klasa podacima na osnovu `cent`
 4. Update `cent`
 5. Ponavljaj dok ne prestane da se menja

In [40]:
from sklearn.decomposition import PCA
from IPython.display import clear_output

def min_max_scaler(knj): # min max scaler skalira vrednosti 1 - 10
    return (knj - knj.min()) / (knj.max() - knj.min())*9+1

def rand_cents(knj, k): # stvara random centroide 
    indices = np.random.choice(knj.shape[0], size=k, replace=False)
    return knj[indices]

def klaster(knj, cents): #stvara od najblizih tacaka klaster
    distances = np.linalg.norm(knj[:, np.newaxis] - cents, axis=2)
    return np.argmin(distances, axis=1)

def geo_sred(knj, klast, k): # odredjuje sredinu novonastalog klastera
    epsilon = 1e-10
    cents = []
    for i in range(k):
        cluster_points = knj[klast == i]
        if len(cluster_points) == 0:
            cluster_points = knj[np.random.choice(knj.shape[0], 1)]
        geo_mean = np.exp(np.log(cluster_points + epsilon).mean(axis=0))
        cents.append(geo_mean)
    return np.vstack(cents)

def klasterizuj(knj, klast, cents, iter): #iscrtava klastere na grafiku real time dok klasterizacija traje
    pca = PCA(n_components=2)
    knj2d = pca.fit_transform(knj)
    cents2d = pca.transform(cents)
    clear_output(wait=True)
    plt.figure(figsize=(7, 5))
    plt.title(f"Iteracija broj {iter}")
    plt.scatter(x=knj2d[:,0], y=knj2d[:,1], c=klast, cmap='tab10', s=10)
    plt.scatter(x=cents2d[:,0], y=cents2d[:,1], c='black', marker='x', s=100)
    plt.show()

def k_means(knj, k, max_iter): # funkcija koja uravlja ostalim funkcijama kako bi se klasterizacija obavila
    knj = min_max_scaler(knj).values if hasattr(knj, 'values') else knj
    cents = rand_cents(knj, k)
    old_cents = np.zeros_like(cents)
    i = 1
    while i < max_iter and not np.allclose(cents, old_cents):
        old_cents = cents
        klast = klaster(knj, cents)
        cents = geo_sred(knj, klast, k)
        klasterizuj(knj, klast, cents, i)
        i += 1
    return klast, cents, knj

Testiranjem različitih atributa za određivanje klastera, primetio sam da `materijal` i `cena` imaju visok nivo uticaja na klastere pa su oni podeljeni vertikalnim linijama. Zbog toga sam uzeo ostale numeričke atribute.

In [28]:
atributi = ['godina', 'materijal', 'cena']
knj = df[atributi].copy()
knj = knj.astype(float)
knj.describe()

,godina,materijal,cena
count,21410.000000,21410.000000,21410.000000
mean,2017.825315,71269.768949,1110.111537
std,5.126895,49960.952220,629.245050
min,1992.000000,1440.000000,249.000000
25%,2015.000000,36400.000000,715.000000
50%,2019.000000,61828.000000,990.000000
75%,2022.000000,92742.000000,1299.000000
max,2024.000000,522038.400000,5489.000000


In [42]:
max_iter = 100
k = 4
klast, centroids, knj_scaled = k_means(knj, k, max_iter)

TypeError: 'module' object is not callable

In [32]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

def plot_clusters(X, labels, title):
    pca = PCA(n_components=2)
    X_2d = pca.fit_transform(X)
    plt.figure(figsize=(6,5))
    plt.scatter(X_2d[:,0], X_2d[:,1], c=labels, cmap='tab10', s=10)
    plt.title(title)
    plt.show()

skaler = MinMaxScaler(feature_range=(1, 10))
knj_sklearn_scaled = skaler.fit_transform(knj)
sklearn_kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
sklearn_kmeans.fit(knj_sklearn_scaled)
sklearn_labels = sklearn_kmeans.labels_

plot_clusters(knj_sklearn_scaled, sklearn_labels, "sklearn KMeans Klasteri")

# === 11. Metrike za obe verzije ===
print("== METRIKE ==")
print(f"My KMeans - Silhouette: {silhouette_score(knj_scaled, klast):.3f}")
print(f"My KMeans - Davies-Bouldin: {davies_bouldin_score(knj_scaled, klast):.3f}")

print(f"sklearn KMeans - Silhouette: {silhouette_score(knj_sklearn_scaled, sklearn_labels):.3f}")
print(f"sklearn KMeans - Davies-Bouldin: {davies_bouldin_score(knj_sklearn_scaled, sklearn_labels):.3f}")

# === 12. Elbow metoda za optimalni k ===
wcss_scaled = []
wcss_no_scaling = []
K_range = range(1, 11)

for k in K_range:
    model_raw = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    model_raw.fit(knj)
    wcss_no_scaling.append(model_raw.inertia_)

    model_scaled = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    model_scaled.fit(knj_sklearn_scaled)
    wcss_scaled.append(model_scaled.inertia_)

plt.figure(figsize=(8,5))
plt.plot(K_range, wcss_no_scaling, 'o-', label='Bez skaliranja')
plt.plot(K_range, wcss_scaled, 'o-', label='Sa skaliranjem (1-10)')
plt.xlabel('Broj klastera (k)')
plt.ylabel('WCSS (Inertia)')
plt.title('Elbow metoda za optimalni k')
plt.legend()
plt.show()

TypeError: 'module' object is not callable